# Kalenjin ASR: Qualitative Evaluation

**Purpose**: Load the fine-tuned model from HuggingFace, build a KenLM language model, and run inference with both greedy and beam search decoding.

**Model**: `RareElf/kalenjin-asr`
**Data**: `data/kln/test.tsv` + `data/kln/clips/`
**LM**: 5-gram KenLM built from `data/kln/train.tsv`

## 1. Setup

In [12]:
import warnings
warnings.filterwarnings('ignore')

import torch
import pandas as pd
import numpy as np
import librosa
import json
import re
import subprocess
from pathlib import Path
from transformers import AutoProcessor, AutoModelForCTC
from pyctcdecode import build_ctcdecoder
from jiwer import wer, cer
from collections import Counter

print(f"PyTorch: {torch.__version__}")
print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

PyTorch: 2.11.0+cpu
Device: cpu


## 2. Load Model from HuggingFace

In [13]:
MODEL_ID = "RareElf/kalenjin-asr"

print(f"Loading model: {MODEL_ID}")
processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModelForCTC.from_pretrained(MODEL_ID)
model.eval()

print(f"\u2713 Model loaded")
print(f"  Parameters: {model.num_parameters():,}")

# Get vocabulary
vocab = processor.tokenizer.get_vocab()
sorted_vocab = sorted(vocab.items(), key=lambda x: x[1])
labels = [k for k, v in sorted_vocab]
print(f"  Vocabulary size: {len(labels)}")
print(f"  Labels: {labels}")

Loading model: RareElf/kalenjin-asr


Loading weights: 100%|██████████| 424/424 [00:00<00:00, 2732.88it/s]


✓ Model loaded
  Parameters: 315,474,595
  Vocabulary size: 35
  Labels: ['|', "'", 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', 'á', 'é', 'í', '[UNK]', '[PAD]', '<s>', '</s>']


## 3. Text Normalization

In [14]:
def normalize_text(text):
    """Apply the same normalization used during training."""
    text = text.lower()
    text = text.replace('\u2018', "'").replace('\u2019', "'").replace('`', "'")
    text = text.replace('ch', 'c').replace('kh', 'k')
    text = re.sub(r"[^a-z' ]", '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

print(normalize_text("Tomo itinye choruet ne chepto iman?"))
print(normalize_text("kiyaat baruet Eng' kasarta ne kimito"))

tomo itinye coruet ne cepto iman
kiyaat baruet eng' kasarta ne kimito


## 4. Build KenLM Language Model from Training Data

In [15]:
DATA_DIR = Path('../data/kln')
CLIPS_DIR = DATA_DIR / 'clips'
LM_DIR = Path('../models/lm')
LM_DIR.mkdir(parents=True, exist_ok=True)

# Load training transcriptions
train_df = pd.read_csv(DATA_DIR / 'train.tsv', sep='\t')
print(f"Training sentences: {len(train_df)}")

# Normalize and save as text corpus
corpus_path = LM_DIR / 'train_corpus.txt'
sentences = []
for text in train_df['sentence'].dropna():
    norm = normalize_text(text)
    if norm.strip():
        sentences.append(norm)

with open(corpus_path, 'w') as f:
    f.write('\n'.join(sentences))

print(f"Normalized sentences: {len(sentences)}")
print(f"Corpus saved to: {corpus_path}")

# Build unigram list for decoder
all_words = []
for s in sentences:
    all_words.extend(s.split())
unigrams = list(set(all_words))
print(f"Unique unigrams: {len(unigrams)}")

Training sentences: 11065
Normalized sentences: 11065
Corpus saved to: ../models/lm/train_corpus.txt
Unique unigrams: 18730


In [17]:
# Build 5-gram ARPA model using KenLM
arpa_path = LM_DIR / 'kalenjin_5gram.arpa'

# Check if lmplz is available
import shutil
lmplz_path = shutil.which('lmplz')

if lmplz_path:
    print(f"Building 5-gram LM with lmplz...")
    result = subprocess.run(
        f'lmplz -o 5 < {corpus_path} > {arpa_path}',
        shell=True, capture_output=True, text=True
    )
    if result.returncode == 0:
        print(f"\u2713 ARPA model saved to: {arpa_path}")
    else:
        print(f"lmplz error: {result.stderr[:200]}")
        arpa_path = None
else:
    print("lmplz not found. Building decoder without ARPA file (unigram only).")
    arpa_path = None

Building 5-gram LM with lmplz...
✓ ARPA model saved to: ../models/lm/kalenjin_5gram.arpa


In [19]:
# Build CTC decoder with language model
print("Building CTC decoder...")

decoder = build_ctcdecoder(
    labels=labels,
    kenlm_model_path=str(arpa_path) if arpa_path and arpa_path.exists() else None,
    unigrams=unigrams,
    alpha=0.6,   # LM weight
    beta=1.0     # word insertion bonus
)

print(f"\u2713 Decoder built")
print(f"  LM: {'5-gram ARPA' if arpa_path and arpa_path.exists() else 'unigram only'}")
print(f"  Alpha: 0.6, Beta: 1.0")

Found entries of length > 1 in alphabet. This is unusual unless style is BPE, but the alphabet was not recognized as BPE type. Is this correct?
Loading the LM will be faster if you build a binary file.
Reading /home/obote/Documents/Hineni/Kalenjin-ASR/models/lm/kalenjin_5gram.arpa
----5---10---15---20---25---30---35---40---45---50---55---60---65---70---75---80---85---90---95--100
****************************************************************************************************


Building CTC decoder...
✓ Decoder built
  LM: 5-gram ARPA
  Alpha: 0.6, Beta: 1.0


## 5. Load Test Data

In [20]:
test_df = pd.read_csv(DATA_DIR / 'test.tsv', sep='\t')
test_df['audio_path'] = test_df['path'].apply(lambda p: CLIPS_DIR / p)
test_df['exists'] = test_df['audio_path'].apply(lambda p: p.exists())
valid_df = test_df[test_df['exists']].reset_index(drop=True)

print(f"Test samples: {len(test_df)}")
print(f"With audio files: {len(valid_df)}")

Test samples: 5685
With audio files: 5685


## 6. Run Inference (Greedy + Beam Search)

In [21]:
N_SAMPLES = 50

results = []
for i in range(min(N_SAMPLES, len(valid_df))):
    row = valid_df.iloc[i]
    audio_path = str(row['audio_path'])
    reference = normalize_text(row['sentence'])
    
    if not reference.strip():
        continue
    
    try:
        # Load audio
        audio, sr = librosa.load(audio_path, sr=16000)
        inputs = processor(audio, sampling_rate=16000, return_tensors='pt', padding=True)
        
        with torch.no_grad():
            logits = model(**inputs).logits
        
        # Greedy decoding
        pred_ids = torch.argmax(logits, dim=-1)
        greedy_pred = processor.batch_decode(pred_ids)[0].lower().strip()
        
        # Beam search with LM
        logits_np = logits[0].cpu().numpy()
        beam_pred = decoder.decode(logits_np, beam_width=100).lower().strip()
        
        # Metrics
        greedy_wer = wer(reference, greedy_pred) if greedy_pred else 1.0
        greedy_cer = cer(reference, greedy_pred) if greedy_pred else 1.0
        beam_wer = wer(reference, beam_pred) if beam_pred else 1.0
        beam_cer = cer(reference, beam_pred) if beam_pred else 1.0
        
        results.append({
            'idx': i,
            'file': row['path'],
            'reference': reference,
            'greedy_pred': greedy_pred,
            'beam_pred': beam_pred,
            'greedy_wer': round(greedy_wer, 4),
            'greedy_cer': round(greedy_cer, 4),
            'beam_wer': round(beam_wer, 4),
            'beam_cer': round(beam_cer, 4)
        })
        
        if i < 10:
            print(f"[{i}] REF:    {reference}")
            print(f"     GREEDY: {greedy_pred}  (WER={greedy_wer:.2f}, CER={greedy_cer:.2f})")
            print(f"     BEAM:   {beam_pred}  (WER={beam_wer:.2f}, CER={beam_cer:.2f})")
            lm_helped = '\u2713 LM helped' if beam_wer < greedy_wer else ('= same' if beam_wer == greedy_wer else '\u2717 LM hurt')
            print(f"     {lm_helped}")
            print()
    except Exception as e:
        print(f"[{i}] Error: {e}")

print(f"\nProcessed {len(results)} samples")

[0] REF:    kamet tinyei lakwengung kenyisiek ata
     GREEDY: kamate tinye lakweng'ung' kenyisiek ata  (WER=0.60, CER=0.14)
     BEAM:   kamete tinye lakweng'ung' kenyisiek ata  (WER=0.60, CER=0.11)
     = same

[1] REF:    komangen ale tos rikci ci ko u no
     GREEDY: komangen ale tos rikchi chi kou no  (WER=0.50, CER=0.09)
     BEAM:   komangen ale tos rikchichi ko u no  (WER=0.25, CER=0.09)
     ✓ LM helped

[2] REF:    uiy ke nem atepto ne yaa ne kakicop
     GREEDY: ui kenam atepto neya ne kokichop  (WER=0.75, CER=0.20)
     BEAM:   ui kenam atepto ne ya ne kokichop  (WER=0.62, CER=0.17)
     ✓ LM helped

[3] REF:    kiiyan kocengei logoiwek eng oldo age
     GREEDY: kiyan kochenge logoiwek eng oldo age  (WER=0.33, CER=0.08)
     BEAM:   kiyan kochenge logoiwek eng' oldo age  (WER=0.50, CER=0.11)
     ✗ LM hurt

[4] REF:    tos iboe nguruonikuk
     GREEDY: tos iboe ngurwonik kuki  (WER=0.67, CER=0.20)
     BEAM:   tos iboengurwonik kuk  (WER=0.67, CER=0.20)
     = same

[5] REF

## 7. Aggregate Metrics: Greedy vs Beam Search

In [22]:
all_refs = [r['reference'] for r in results]
all_greedy = [r['greedy_pred'] for r in results]
all_beam = [r['beam_pred'] for r in results]

greedy_overall_wer = wer(all_refs, all_greedy)
greedy_overall_cer = cer(all_refs, all_greedy)
beam_overall_wer = wer(all_refs, all_beam)
beam_overall_cer = cer(all_refs, all_beam)

print("=" * 60)
print(f"RESULTS ({len(results)} samples)")
print("=" * 60)
print(f"{'Metric':<20} {'Greedy':>12} {'Beam+LM':>12} {'Improvement':>12}")
print("-" * 60)
print(f"{'WER':<20} {greedy_overall_wer*100:>11.2f}% {beam_overall_wer*100:>11.2f}% {(greedy_overall_wer-beam_overall_wer)*100:>+11.2f}pp")
print(f"{'CER':<20} {greedy_overall_cer*100:>11.2f}% {beam_overall_cer*100:>11.2f}% {(greedy_overall_cer-beam_overall_cer)*100:>+11.2f}pp")
print()

lm_helped = sum(1 for r in results if r['beam_wer'] < r['greedy_wer'])
lm_same = sum(1 for r in results if r['beam_wer'] == r['greedy_wer'])
lm_hurt = sum(1 for r in results if r['beam_wer'] > r['greedy_wer'])
print(f"LM helped: {lm_helped}/{len(results)} ({lm_helped/len(results)*100:.1f}%)")
print(f"LM same:   {lm_same}/{len(results)} ({lm_same/len(results)*100:.1f}%)")
print(f"LM hurt:   {lm_hurt}/{len(results)} ({lm_hurt/len(results)*100:.1f}%)")

RESULTS (50 samples)
Metric                     Greedy      Beam+LM  Improvement
------------------------------------------------------------
WER                        73.64%       60.00%      +13.64pp
CER                        18.33%       16.14%       +2.19pp

LM helped: 28/50 (56.0%)
LM same:   16/50 (32.0%)
LM hurt:   6/50 (12.0%)


## 8. Best and Worst Predictions

In [23]:
sorted_by_cer = sorted(results, key=lambda x: x['beam_cer'])

print("=== TOP 10 BEST (lowest CER, beam) ===")
for r in sorted_by_cer[:10]:
    print(f"  [{r['idx']}] CER={r['beam_cer']:.2f} WER={r['beam_wer']:.2f}")
    print(f"    REF:  {r['reference']}")
    print(f"    BEAM: {r['beam_pred']}")
    print()

print("\n=== TOP 10 WORST (highest CER, beam) ===")
for r in sorted_by_cer[-10:]:
    print(f"  [{r['idx']}] CER={r['beam_cer']:.2f} WER={r['beam_wer']:.2f}")
    print(f"    REF:  {r['reference']}")
    print(f"    BEAM: {r['beam_pred']}")
    print()

=== TOP 10 BEST (lowest CER, beam) ===
  [40] CER=0.03 WER=0.14
    REF:  uni mengen kiy age tugul akobo kiooe
    BEAM: uni mengen kiy age tugul akobo kioo

  [17] CER=0.03 WER=0.20
    REF:  cesawil ne kinte mising korona
    BEAM: cesawil ne kinde mising korona

  [25] CER=0.03 WER=0.10
    REF:  sait ake tukul kemi twai kou tumdap comyet eng' muguleldanyu
    BEAM: sait ake tukul kemi twai kou tumdap chamyet eng' muguleldanyu

  [47] CER=0.05 WER=0.44
    REF:  inyolu kotepso kiboitinik eng oret necamat eng taitab kamuktaindet
    BEAM: inyolu kotepso kiboitinik eng oret ne chamat eng tait ab kamuktaindet

  [41] CER=0.06 WER=0.50
    REF:  mumatiroto kukeny
    BEAM: mumatirotok kukeny

  [19] CER=0.06 WER=0.45
    REF:  barakutap coto ko afisayat age tugul eng serkali kecikil mogornondit netinyei
    BEAM: barakutab choto ko afisayat age tugul eng serikali kechikil mokornondit netinyei

  [49] CER=0.07 WER=0.57
    REF:  ngalekci ce tuten kokitoret kosir kitabut nyi
    BEAM: nga

## 9. LM Impact Examples

In [24]:
# Show cases where LM made the biggest improvement
lm_improvements = sorted(results, key=lambda r: r['greedy_wer'] - r['beam_wer'], reverse=True)

print("=== TOP 10 LM IMPROVEMENTS ===")
for r in lm_improvements[:10]:
    improvement = r['greedy_wer'] - r['beam_wer']
    if improvement <= 0:
        break
    print(f"  [{r['idx']}] WER: {r['greedy_wer']:.2f} -> {r['beam_wer']:.2f} ({improvement:+.2f})")
    print(f"    REF:    {r['reference']}")
    print(f"    GREEDY: {r['greedy_pred']}")
    print(f"    BEAM:   {r['beam_pred']}")
    print()

=== TOP 10 LM IMPROVEMENTS ===
  [37] WER: 1.67 -> 1.00 (+0.67)
    REF:    bukonyoru camanenyi jumamos
    GREEDY: po konyoru chamanenyi ju mamos
    BEAM:   po konyoru chamanenyi jumamos

  [18] WER: 1.00 -> 0.38 (+0.62)
    REF:    camiet ne ng'un ko u atindeyot nebo baibaiet
    GREEDY: chomyet nengun kou afingeyos nepo poipoyi kuput g
    BEAM:   chomnyet ne ng'un ko u atindeyot nebo boiboiye koput

  [49] WER: 1.00 -> 0.57 (+0.43)
    REF:    ngalekci ce tuten kokitoret kosir kitabut nyi
    GREEDY: ngalek chi chetuten ko kitoret kosir kitabutie
    BEAM:   ngalek chi ce tuten ko kitoret kosir kitabut nyi

  [17] WER: 0.60 -> 0.20 (+0.40)
    REF:    cesawil ne kinte mising korona
    GREEDY: cheasawid ne kinde mising' korona
    BEAM:   cesawil ne kinde mising korona

  [13] WER: 1.00 -> 0.62 (+0.38)
    REF:    nereeekap mungaranyii kobonuu kesutik ce bo kakiletap kei
    GREEDY: nerek ab mmung'arengyu kobunu kesutik chebo kokiletab kee
    BEAM:   neret ab mung'arenik kobunu k

## 10. Error Pattern Analysis

In [25]:
substitutions = Counter()
insertions = Counter()
deletions = Counter()

for r in results:
    ref_chars = list(r['reference'])
    pred_chars = list(r['beam_pred'])
    
    min_len = min(len(ref_chars), len(pred_chars))
    for j in range(min_len):
        if ref_chars[j] != pred_chars[j]:
            substitutions[(ref_chars[j], pred_chars[j])] += 1
    
    if len(pred_chars) > len(ref_chars):
        for c in pred_chars[min_len:]:
            insertions[c] += 1
    elif len(ref_chars) > len(pred_chars):
        for c in ref_chars[min_len:]:
            deletions[c] += 1

print("=== TOP 15 CHARACTER SUBSTITUTIONS ===")
for (ref_c, pred_c), count in substitutions.most_common(15):
    print(f"  '{ref_c}' -> '{pred_c}': {count}")

print("\n=== TOP 10 DELETED CHARACTERS ===")
for c, count in deletions.most_common(10):
    print(f"  '{c}': {count}")

print("\n=== TOP 10 INSERTED CHARACTERS ===")
for c, count in insertions.most_common(10):
    print(f"  '{c}': {count}")

=== TOP 15 CHARACTER SUBSTITUTIONS ===
  ' ' -> 'k': 33
  'k' -> ' ': 25
  't' -> ' ': 24
  'e' -> ' ': 23
  ' ' -> 'a': 22
  ' ' -> 'e': 18
  'i' -> 'k': 18
  'k' -> 'i': 17
  ' ' -> 'i': 17
  'n' -> 'e': 17
  'o' -> 'k': 16
  'o' -> ' ': 16
  ' ' -> 'o': 15
  'i' -> ' ': 15
  'a' -> 'k': 13

=== TOP 10 DELETED CHARACTERS ===
  'k': 5
  'i': 5
  'e': 5
  'o': 3
  'a': 3
  't': 2
  ' ': 2
  'c': 2
  'r': 2
  'p': 1

=== TOP 10 INSERTED CHARACTERS ===
  'e': 8
  'k': 7
  'i': 7
  'n': 6
  't': 5
  'o': 5
  'a': 4
  'u': 4
  'y': 4
  'g': 3


## 11. Save Results for Paper

In [26]:
output = {
    'model': MODEL_ID,
    'n_samples': len(results),
    'greedy_wer': greedy_overall_wer,
    'greedy_cer': greedy_overall_cer,
    'beam_wer': beam_overall_wer,
    'beam_cer': beam_overall_cer,
    'lm_config': {'alpha': 0.6, 'beta': 1.0, 'beam_width': 100},
    'predictions': results
}

with open('../processed_data/qualitative_results.json', 'w') as f:
    json.dump(output, f, indent=2, default=str)

print(f"Results saved to processed_data/qualitative_results.json")

# Print LaTeX table
print("\n=== LATEX TABLE ===")
print(r"\begin{table}[!t]")
print(r"\caption{Qualitative Prediction Examples (Beam Search + KenLM)}")
print(r"\label{tab:predictions}")
print(r"\centering")
print(r"\scriptsize")
print(r"\begin{tabular}{|p{0.30\columnwidth}|p{0.30\columnwidth}|c|c|}")
print(r"\hline")
print(r"\textbf{Reference} & \textbf{Prediction} & \textbf{WER} & \textbf{CER} \\")
print(r"\hline")
print(r"\hline")

examples = sorted_by_cer[:5] + sorted_by_cer[-5:]
for r in examples:
    ref = r['reference'].replace('&', '\\&').replace('%', '\\%')
    pred = r['beam_pred'].replace('&', '\\&').replace('%', '\\%')
    print(f"{ref} & {pred} & {r['beam_wer']:.2f} & {r['beam_cer']:.2f} \\\\")
    print(r"\hline")

print(r"\end{tabular}")
print(r"\end{table}")

Results saved to processed_data/qualitative_results.json

=== LATEX TABLE ===
\begin{table}[!t]
\caption{Qualitative Prediction Examples (Beam Search + KenLM)}
\label{tab:predictions}
\centering
\scriptsize
\begin{tabular}{|p{0.30\columnwidth}|p{0.30\columnwidth}|c|c|}
\hline
\textbf{Reference} & \textbf{Prediction} & \textbf{WER} & \textbf{CER} \\
\hline
\hline
uni mengen kiy age tugul akobo kiooe & uni mengen kiy age tugul akobo kioo & 0.14 & 0.03 \\
\hline
cesawil ne kinte mising korona & cesawil ne kinde mising korona & 0.20 & 0.03 \\
\hline
sait ake tukul kemi twai kou tumdap comyet eng' muguleldanyu & sait ake tukul kemi twai kou tumdap chamyet eng' muguleldanyu & 0.10 & 0.03 \\
\hline
inyolu kotepso kiboitinik eng oret necamat eng taitab kamuktaindet & inyolu kotepso kiboitinik eng oret ne chamat eng tait ab kamuktaindet & 0.44 & 0.05 \\
\hline
mumatiroto kukeny & mumatirotok kukeny & 0.50 & 0.06 \\
\hline
muc kukerpwonikab cii korona akubar & imuch ko kekwonikabchii korona akob